# Estudo: Floresta Aleatória para Predição de Desempenho Acadêmico

Este notebook implementa e avalia um modelo de **Floresta Aleatória (Random Forest)** para classificar o desempenho acadêmico dos estudantes de uma turma média de ensino médio público no Brasil (dados sintéticos).

**Variável alvo:** `situacao` (Aprovado, Em Risco, Reprovado)

## Etapas:
1. Carregamento e exploração dos dados
2. Pré-processamento (One-Hot Encoding + Feature Engineering)
3. Balanceamento com SMOTE
4. Otimização de hiperparâmetros (GridSearchCV)
5. Treinamento e avaliação do modelo
6. Comparação de hiperparâmetros
7. Importância dos atributos e interpretação

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*")
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from models import preprocess, feature_engineering, split_and_balance, NOMINAL_COLS, BINARY_COLS, LOW_IMPORTANCE_FEATURES

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("Bibliotecas carregadas com sucesso!")

## 1. Carregamento e Exploração dos Dados

In [ ]:
df = pd.read_csv("../data/dados_academicos.csv")
print(f"Shape: {df.shape}")
print(f"\nColunas: {list(df.columns)}")
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Distribuição da variável alvo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

contagem = df["situacao"].value_counts()
colors = ["#2ecc71", "#f39c12", "#e74c3c"]

axes[0].bar(contagem.index, contagem.values, color=colors)
axes[0].set_title("Distribuição da Variável Alvo")
axes[0].set_ylabel("Quantidade")
for i, v in enumerate(contagem.values):
    axes[0].text(i, v + 3, str(v), ha="center", fontweight="bold")

axes[1].pie(contagem.values, labels=contagem.index, autopct="%1.1f%%", colors=colors)
axes[1].set_title("Proporção das Classes")

plt.tight_layout()
plt.show()

## 2. Pré-processamento dos Dados (One-Hot Encoding + Feature Engineering)

In [ ]:
# Verificar valores nulos
print("Valores nulos por coluna:")
print(df.isnull().sum())
print(f"\nTotal de valores nulos: {df.isnull().sum().sum()}")

In [ ]:
# Pré-processamento usando o módulo models.py:
# - Feature Engineering (variáveis derivadas)
# - One-Hot Encoding para variáveis nominais (estado_civil, renda_per_capita, cor_raca)
# - Label Encoding para variáveis binárias (Sim/Nao)
# - Remoção de features de baixa importância

X, y, encoders_info, le_target = preprocess(df)

print(f"Features após pré-processamento: {X.shape[1]}")
print(f"\nColunas resultantes:")
for i, col in enumerate(X.columns):
    print(f"  {i+1}. {col}")
print(f"\nVariáveis removidas (baixa importância): {LOW_IMPORTANCE_FEATURES}")
print(f"Variáveis com One-Hot Encoding: {NOMINAL_COLS}")
print(f"\nTarget classes: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")
print(f"\nDistribuição do target:\n{pd.Series(y).value_counts()}")

In [ ]:
# Separar features e target - já feito pelo preprocess()
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

## 3. Balanceamento com SMOTE

In [ ]:
# Divisão treino/teste (70/30) e SMOTE no treino
X_train, X_test, y_train, y_test, X_train_balanced, y_train_balanced = split_and_balance(X, y)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste:  {X_test.shape[0]} amostras")
print(f"Treino após SMOTE: {X_train_balanced.shape[0]} amostras")
print(f"\nDistribuição no treino (original):\n{pd.Series(y_train).value_counts()}")
print(f"\nDistribuição no treino (SMOTE):\n{pd.Series(y_train_balanced).value_counts()}")
print(f"\nDistribuição no teste:\n{pd.Series(y_test).value_counts()}")

In [ ]:
# Aplicar SMOTE apenas no conjunto de treinamento (já feito acima via split_and_balance)
print(f"Treino antes do SMOTE: {X_train.shape[0]} amostras")
print(f"Treino após SMOTE:     {X_train_balanced.shape[0]} amostras")
print(f"\nDistribuição após SMOTE:\n{pd.Series(y_train_balanced).value_counts()}")

## 4. Otimização de Hiperparâmetros (GridSearchCV)

In [ ]:
# GridSearchCV com SMOTE dentro de cada fold
pipeline_grid = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [8, 10, 12, 15, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 3, 5],
}

cv_grid = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline_grid,
    param_grid,
    cv=cv_grid,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

best_params = {k.replace("classifier__", ""): v for k, v in grid_search.best_params_.items()}
print(f"\nMelhores hiperparâmetros encontrados:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"\nMelhor F1-Score (CV 5-fold): {grid_search.best_score_:.4f}")

## 5. Treinamento e Avaliação do Modelo

In [ ]:
# Treinar modelo final com melhores hiperparâmetros
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1, **best_params)
rf_model.fit(X_train_balanced, y_train_balanced)

# Predições
y_pred = rf_model.predict(X_test)

# Acurácia
acc = accuracy_score(y_test, y_pred)
print(f"Acurácia no conjunto de teste: {acc:.4f} ({acc*100:.2f}%)")
print(f"\nHiperparâmetros utilizados: {best_params}")

In [ ]:
# Relatório de classificação
target_names = le_target.classes_
print("Relatório de Classificação:")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap="Greens", values_format="d")
ax.set_title("Matriz de Confusão - Floresta Aleatória")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Validação Cruzada k-fold (k=10) com SMOTE DENTRO de cada fold
# Isso evita data leakage: dados sintéticos não vazam para validação
# ============================================================

# Pipeline: SMOTE + Floresta Aleatória (com melhores hiperparâmetros)
pipeline_rf = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1, **best_params)),
])

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Múltiplas métricas conforme descrito na metodologia do TCC
scoring = {
    "accuracy": "accuracy",
    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",
}

# Cross-validate nos dados ORIGINAIS (sem SMOTE prévio)
cv_results = cross_validate(
    pipeline_rf, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True
)

print("=" * 70)
print("VALIDAÇÃO CRUZADA (k=10) - FLORESTA ALEATÓRIA")
print("SMOTE aplicado dentro de cada fold (sem data leakage)")
print("=" * 70)

for metric_name in scoring:
    test_scores = cv_results[f"test_{metric_name}"]
    train_scores = cv_results[f"train_{metric_name}"]
    print(f"\n{metric_name.upper()}:")
    print(f"  Treino:    {train_scores.mean():.4f} (+/- {train_scores.std():.4f})")
    print(f"  Validação: {test_scores.mean():.4f} (+/- {test_scores.std():.4f})")
    print(f"  Por fold:  {np.round(test_scores, 4)}")

# Armazenar para o resumo final
cv_acc_mean = cv_results["test_accuracy"].mean()
cv_acc_std = cv_results["test_accuracy"].std()
cv_f1_mean = cv_results["test_f1_weighted"].mean()
cv_f1_std = cv_results["test_f1_weighted"].std()

# Gráfico comparativo treino vs validação por fold
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_list = list(scoring.keys())
colors_train = "#27ae60"
colors_val = "#e74c3c"

for idx, metric_name in enumerate(metrics_list):
    ax = axes[idx // 2][idx % 2]
    folds = range(1, 11)
    test_scores = cv_results[f"test_{metric_name}"]
    train_scores = cv_results[f"train_{metric_name}"]

    ax.plot(folds, train_scores, "o-", color=colors_train,
            label=f"Treino (média={train_scores.mean():.4f})")
    ax.plot(folds, test_scores, "s-", color=colors_val,
            label=f"Validação (média={test_scores.mean():.4f})")
    ax.set_xlabel("Fold")
    ax.set_ylabel("Score")
    ax.set_title(f"{metric_name.replace('_', ' ').title()}")
    ax.set_xticks(folds)
    ax.legend(fontsize=8)
    ax.set_ylim(0.4, 1.05)
    ax.grid(True, alpha=0.3)

plt.suptitle("Validação Cruzada (k=10) - Floresta Aleatória\n(SMOTE dentro de cada fold, hiperparâmetros otimizados)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Comparação de Hiperparâmetros (Número de Árvores)

In [ ]:
# Testar diferentes números de árvores (mantendo demais hiperparâmetros otimizados)
n_estimators_range = [10, 50, 100, 200, 300, 500]
results = []

# Pegar hiperparâmetros otimizados (exceto n_estimators)
params_without_n_est = {k: v for k, v in best_params.items() if k != "n_estimators"}

for n_est in n_estimators_range:
    rf_temp = RandomForestClassifier(
        n_estimators=n_est,
        random_state=42,
        n_jobs=-1,
        **params_without_n_est,
    )
    rf_temp.fit(X_train_balanced, y_train_balanced)
    y_pred_temp = rf_temp.predict(X_test)
    acc_temp = accuracy_score(y_test, y_pred_temp)
    f1_temp = f1_score(y_test, y_pred_temp, average="weighted")
    results.append({"n_estimators": n_est, "accuracy": acc_temp, "f1_weighted": f1_temp})
    print(f"n_estimators={n_est:>4d} | Acurácia={acc_temp:.4f} | F1={f1_temp:.4f}")

df_results = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_results["n_estimators"], df_results["accuracy"], "o-", label="Acurácia")
ax.plot(df_results["n_estimators"], df_results["f1_weighted"], "s-", label="F1-Score")
ax.set_xlabel("Número de Árvores (n_estimators)")
ax.set_ylabel("Score")
ax.set_title("Desempenho vs. Número de Árvores")
ax.legend()
ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.show()

## 7. Importância dos Atributos e Interpretação

In [ ]:
# Importância final dos atributos
feature_importance_final = pd.DataFrame({
    "atributo": X.columns,
    "importancia": rf_model.feature_importances_
}).sort_values("importancia", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=feature_importance_final, x="importancia", y="atributo", hue="atributo", palette="magma", legend=False)
plt.title("Importância dos Atributos - Modelo Final (Floresta Aleatória)")
plt.xlabel("Importância")
plt.ylabel("Atributo")
plt.tight_layout()
plt.show()

print("\nRanking de importância:")
print(feature_importance_final.to_string(index=False))

In [ ]:
# Correlação entre features (usando X que já está codificado)
plt.figure(figsize=(14, 10))
corr_matrix = X.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Matriz de Correlação entre Features")
plt.tight_layout()
plt.show()

## 8. Resumo dos Resultados

In [ ]:
print("=" * 60)
print("RESUMO - FLORESTA ALEATÓRIA")
print("=" * 60)
print(f"Total de amostras:           {len(df)}")
print(f"Amostras de treino:          {X_train.shape[0]} (original) -> {X_train_balanced.shape[0]} (SMOTE)")
print(f"Amostras de teste:           {X_test.shape[0]}")
print(f"Número de features:          {X.shape[1]}")
print(f"Hiperparâmetros otimizados:  {best_params}")
print(f"Acurácia (teste holdout):    {acc:.4f}")
print(f"Acurácia média (CV k=10):    {cv_acc_mean:.4f} (+/- {cv_acc_std:.4f})")
print(f"F1-Score médio (CV k=10):    {cv_f1_mean:.4f} (+/- {cv_f1_std:.4f})")
print(f"Precision média (CV k=10):   {cv_results['test_precision_weighted'].mean():.4f}")
print(f"Recall médio (CV k=10):      {cv_results['test_recall_weighted'].mean():.4f}")
print(f"\nTop 5 atributos mais importantes:")
for _, row in feature_importance_final.head(5).iterrows():
    print(f"  - {row['atributo']}: {row['importancia']:.4f}")
print("=" * 60)